# Claude와 calculator tool 사용하기
이 예제에서는 사용자 입력에 따라 산술 연산을 수행할 수 있는 간단한 calculator tool을 Claude에 제공하는 방법을 보여 줍니다. calculator tool을 정의하고 Claude가 이를 사용하여 수학 문제를 해결하는 방법을 보여 줍니다.


## 1단계: environment 설정하기

먼저 필요한 library를 설치하고 Claude API client를 설정하겠습니다.


In [ ]:
%pip install anthropic

In [11]:
from anthropic import Anthropic

client = Anthropic()
MODEL_NAME = "claude-opus-4-8"

## 2단계: calculator tool 정의하기
기본 산술 연산을 수행할 수 있는 간단한 calculator tool을 정의합니다. 수학식을 입력받아 결과를 반환합니다.

출력된 식에 ```eval```을 호출한다는 점에 유의하세요. 이는 좋지 않은 관행이며 일반적으로 사용해서는 안 되지만 여기서는 시연 목적으로 수행합니다.


In [ ]:
import re


def calculate(expression):
    # 수나 연산자가 아닌 문자를 expression에서 모두 제거
    expression = re.sub(r"[^0-9+\-*/().]", "", expression)

    try:
        # 내장 eval() 함수로 expression 평가
        # 참고: 여기서 eval은 시연 목적으로만 사용합니다.
        # Production에서는 ast.literal_eval이나 math parser 같은 더 안전한 대안을 사용하세요.
        result = eval(expression)  # noqa: S307
        return str(result)
    except (SyntaxError, ZeroDivisionError, NameError, TypeError, OverflowError):
        return "Error: Invalid expression"


tools = [
    {
        "name": "calculator",
        "description": "A simple calculator that performs basic arithmetic operations.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The mathematical expression to evaluate (e.g., '2 + 3 * 4').",
                }
            },
            "required": ["expression"],
        },
    }
]

이 예제에서는 수학식을 입력받아 regular expression으로 숫자와 operator가 아닌 문자를 제거한 뒤 built-in eval() 함수로 평가하는 calculate 함수를 정의합니다. 평가가 성공하면 결과를 문자열로 반환합니다. 평가 중 오류가 발생하면 오류 message를 반환합니다.

그다음 string 타입의 expression property 하나를 기대하는 input schema로 calculator tool을 정의합니다.


## 3단계: Claude와 상호 작용하기
이제 Claude가 calculator tool을 사용하여 수학 문제를 해결하는 방법을 살펴보겠습니다.


In [36]:
def process_tool_call(tool_name, tool_input):
    if tool_name == "calculator":
        return calculate(tool_input["expression"])


def chat_with_claude(user_message):
    print(f"\n{'=' * 50}\nUser Message: {user_message}\n{'=' * 50}")

    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        messages=[{"role": "user", "content": user_message}],
        tools=tools,
    )

    print("\nInitial Response:")
    print(f"Stop Reason: {message.stop_reason}")
    print(f"Content: {message.content}")

    if message.stop_reason == "tool_use":
        tool_use = next(block for block in message.content if block.type == "tool_use")
        tool_name = tool_use.name
        tool_input = tool_use.input

        print(f"\nTool Used: {tool_name}")
        print(f"Tool Input: {tool_input}")

        tool_result = process_tool_call(tool_name, tool_input)

        print(f"Tool Result: {tool_result}")

        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=4096,
            messages=[
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": message.content},
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "tool_result",
                            "tool_use_id": tool_use.id,
                            "content": tool_result,
                        }
                    ],
                },
            ],
            tools=tools,
        )
    else:
        response = message

    final_response = next(
        (block.text for block in response.content if hasattr(block, "text")),
        None,
    )
    print(response.content)
    print(f"\nFinal Response: {final_response}")

    return final_response

## 4단계: 직접 사용해 보세요!

이제 calculator에 접근할 수 있으므로 Claude에 몇 가지 예제 수학 질문을 주겠습니다.


In [37]:
chat_with_claude("What is the result of 1,984,135 * 9,343,116?")
chat_with_claude("Calculate (12851 - 593) * 301 + 76")
chat_with_claude("What is 15910385 divided by 193053?")


User Message: What is the result of 1,984,135 * 9,343,116?

Initial Response:
Stop Reason: tool_use
Content: [ContentBlock(text='<thinking>\nThe calculator function is the relevant tool to answer this request, since it involves evaluating a mathematical expression.\n\nThe required parameter for the calculator function is:\nexpression: The mathematical expression to evaluate.\n\nThe human has directly provided the full expression to evaluate in their request: "1,984,135 * 9,343,116". This contains all the information needed for the required expression parameter.\n\nSo I have the necessary information to invoke the calculator tool.\n</thinking>', type='text'), ContentBlockToolUse(id='toolu_01V2mzqp5qkB5QucRFjJUJLD', input={'expression': '1984135 * 9343116'}, name='calculator', type='tool_use')]

Tool Used: calculator
Tool Input: {'expression': '1984135 * 9343116'}
Tool Result: 18538003464660
[ContentBlock(text='Therefore, the result of 1,984,135 * 9,343,116 is 18,538,003,464,660.', type

'So 15910385 divided by 193053 equals 82.41459599177428.'